## 2.b. [10 points] Run the code on one of your favorite classification datasets. Make any changes to suit a classification task. You can use Spark installed on your laptop. Determine the train and test accuracy.

In [ ]:
from pyspark.sql import SparkSession

# Start Spark session
spark = SparkSession.builder.appName("BreastCancerLR").getOrCreate()
sc = spark.sparkContext
print(" Spark session started")


 Spark session started


In [ ]:
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.classification import LogisticRegressionWithSGD
from pyspark.mllib.linalg import Vectors

# Load dataset from uploaded CSV
df = spark.read.csv("breast_cancer.csv", header=True, inferSchema=True)
feature_cols = [c for c in df.columns if c != "label"]

# Convert to LabeledPoint RDD
def row_to_lp(row):
    feats = [float(getattr(row, c)) for c in feature_cols]
    return LabeledPoint(float(row["label"]), Vectors.dense(feats))

rdd = df.rdd.map(row_to_lp)
train_rdd, test_rdd = rdd.randomSplit([0.75, 0.25], seed=42)

# Train "naive" LR with SGD
model = LogisticRegressionWithSGD.train(train_rdd, iterations=200, step=0.1)

# Evaluate accuracies
def accuracy(rdd):
    preds = rdd.map(lambda lp: (float(model.predict(lp.features)), lp.label))
    return preds.filter(lambda pl: pl[0] == pl[1]).count() / rdd.count()

train_acc = accuracy(train_rdd)
test_acc = accuracy(test_rdd)
print(f" [RDD / MLlib + SGD] Train accuracy: {train_acc:.4f}")
print(f" [RDD / MLlib + SGD] Test accuracy: {test_acc:.4f}")


 [RDD / MLlib + SGD] Train accuracy: 0.8979
 [RDD / MLlib + SGD] Test accuracy: 0.8851


## 2.c. [10 points] Use LogisticRegression from pyspark.ml.classification to determine the train and test accuracy on the same dataset. Compare the two approaches.



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# start spark session
spark = SparkSession.builder.appName("Part2c_LogReg_DF").getOrCreate()

# load breast cancer dataset from sklearn
try:
    from sklearn.datasets import load_breast_cancer
    import pandas as pd
    data = load_breast_cancer(as_frame=True)
    pdf = data.frame.copy()
    pdf["target"] = pdf["target"].astype(int)
    df = spark.createDataFrame(pdf)
    print("Dataset loaded successfully from sklearn.")
except Exception as e:
    print("Error loading sklearn dataset:", e)
    df = spark.read.csv("breast_cancer.csv", header=True, inferSchema=True)

# define label and feature columns
label_col = "target"
feature_cols = [c for c in df.columns if c != label_col]

# cast label to double type
df = df.select(*(feature_cols + [col(label_col).cast("double").alias(label_col)]))

# show class distribution
print("Class distribution:")
df.groupBy(label_col).count().orderBy(label_col).show()

# split data into train and test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# create feature assembler and scaler
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# initialize logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=200)

# create pipeline for all stages
pipeline = Pipeline(stages=[assembler, scaler, lr])

# train model
model = pipeline.fit(train_df)
lr_model = model.stages[-1]

# print model summary
print("\n=== Model Summary ===")
print("Intercept:", lr_model.intercept)
print("Num features:", len(lr_model.coefficients))

# make predictions
train_pred = model.transform(train_df)
test_pred = model.transform(test_df)

# evaluate accuracy
evaluator = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName="accuracy")
train_acc = evaluator.evaluate(train_pred)
test_acc = evaluator.evaluate(test_pred)

print("\n=== Accuracy ===")
print("Train accuracy:", round(train_acc, 4))
print("Test accuracy:", round(test_acc, 4))

# show few predictions
print("\nSample predictions:")
test_pred.select("prediction", "probability", label_col).show(5, truncate=False)

# stop spark session
spark.stop()


Dataset loaded successfully from sklearn.
Class distribution:
+------+-----+
|target|count|
+------+-----+
|   0.0|  212|
|   1.0|  357|
+------+-----+


=== Model Summary ===
Intercept: -288.01026091810064
Num features: 30

=== Accuracy ===
Train accuracy: 1.0
Test accuracy: 0.9368

Sample predictions:
+----------+-----------+------+
|prediction|probability|target|
+----------+-----------+------+
|1.0       |[0.0,1.0]  |1.0   |
|1.0       |[0.0,1.0]  |1.0   |
|1.0       |[0.0,1.0]  |1.0   |
|1.0       |[0.0,1.0]  |1.0   |
|1.0       |[0.0,1.0]  |1.0   |
+----------+-----------+------+
only showing top 5 rows



### Comparison: RDD vs DataFrame Logistic Regression

The RDD based logistic regression use low level functions and needs more manual work to prepare the data, while the DataFrame API use pipelines which makes it much easier and clean to handle.  
In RDD version, the model use SGD optimizer which is slow and need tuning for learning rate, but in DataFrame version it use LBFGS which converge faster and more stable.  
For features, RDD need LabeledPoint objects but in DataFrame we can just use VectorAssembler and StandardScaler to combine and scale everything automatically.  

When comparing the results on Breast Cancer dataset:  
- **RDD (SGD)** — Train accuracy: *0.8979*, Test accuracy: *0.8851*  
- **DataFrame (LBFGS)** — Train accuracy: *1.0*, Test accuracy: *0.9368*  

The new DataFrame approach clearly perform better and also took less effort to code.  
Overall, the Pipeline API feels more modern and powerful, while RDD one is more old style and not really used much now.
